# L8d Lab: Cross-Validation and Residual Model Checking

A validation score tells us **how well** a model generalizes; a residual plot can suggest **why** it misses. In this lab, those are consecutive steps in one modeling workflow.

> **Learning objectives**
>
> - Select a ridge penalty with reproducible cross-validation and training-only scaling.
> - Diagnose a missing nonlinear feature from structured residuals.
> - Add a feature justified by the diagnostic, then repeat cross-validation.
> - Separate exploratory model revision from a final claim about unseen data.


## Setup

The local setup activates the pinned course environment and loads the tested ridge and cross-validation functions from [`../src/Week08Core.jl`](../src/Week08Core.jl).


In [ ]:
include(joinpath(@__DIR__, "Include.jl"))


## 1. Start with a plausible but incomplete feature map

The response contains ordinary linear effects plus a centered quadratic contribution from feature 4. We initially give the model only the eight linear columns in `X`. This is a controlled example of model misspecification: the fitting algorithm is working correctly, but the chosen feature map cannot represent the full signal.


In [ ]:
rng = MersenneTwister(5800)
n, p = 180, 8
X = randn(rng, n, p)
quadratic_feature = X[:, 4].^2
quadratic_signal = 2.0 .* (quadratic_feature .- mean(quadratic_feature))
y = 4.0 .+ 2.0 .* X[:, 1] .- 1.5 .* X[:, 2] .+ 0.8 .* X[:, 3] .+
    quadratic_signal .+ 0.45 .* randn(rng, n)

lambdas = vcat(0.0, 10.0 .^ range(-4, 3; length = 15))
baseline_cv = cross_validate_ridge(X, y, lambdas; k = 6, seed = 5800)
(selected_lambda = baseline_cv.best_lambda, validation_rmse = minimum(baseline_cv.mean_rmse))


## 2. Fit the selected model and inspect its residuals

Cross-validation has selected the regularization strength without using a validation fold to estimate its own scaling parameters. We now refit that selected model on all observations for diagnosis. The residual is $e_i=y_i-\hat y_i$. If the linear feature map were adequate, the residual cloud should not retain a systematic relationship with an available predictor.


In [ ]:
baseline_scaled = standardize_train_test(X, X)
baseline_fit = ridge_fit(baseline_scaled.train, y, baseline_cv.best_lambda)
baseline_residual = y - baseline_fit.predictions
baseline_metrics = model_metrics(y, baseline_fit.predictions)

scatter(X[:, 4], baseline_residual; xlabel = "feature 4", ylabel = "residual",
    label = false, alpha = 0.65, title = "Linear feature map leaves curvature")
hline!([0.0]; color = :black, linewidth = 2, label = false)


## 3. Revise the feature map, then re-run model selection

The U-shaped residual pattern motivates adding $x_4^2$; it does not motivate blindly increasing polynomial degree everywhere. Because the model family changed, the old value of $\lambda$ is no longer automatically appropriate. We therefore repeat the entire cross-validation calculation for the augmented feature matrix.


In [ ]:
X_augmented = hcat(X, quadratic_feature)
augmented_cv = cross_validate_ridge(X_augmented, y, lambdas; k = 6, seed = 5800)
augmented_scaled = standardize_train_test(X_augmented, X_augmented)
augmented_fit = ridge_fit(augmented_scaled.train, y, augmented_cv.best_lambda)
augmented_residual = y - augmented_fit.predictions
augmented_metrics = model_metrics(y, augmented_fit.predictions)

comparison = DataFrame(
    model = ["linear features", "linear + x4^2"],
    selected_lambda = [baseline_cv.best_lambda, augmented_cv.best_lambda],
    cv_rmse = [minimum(baseline_cv.mean_rmse), minimum(augmented_cv.mean_rmse)],
    refit_rmse = [baseline_metrics.rmse, augmented_metrics.rmse],
)
pretty_table(comparison)
comparison


In [ ]:
cv_plot = plot(lambdas[2:end], baseline_cv.mean_rmse[2:end]; xscale = :log10,
    marker = :circle, xlabel = "ridge penalty λ", ylabel = "mean validation RMSE",
    label = "linear features", title = "Cross-validation must be repeated")
plot!(cv_plot, lambdas[2:end], augmented_cv.mean_rmse[2:end]; marker = :square,
    label = "linear + x4²")

residual_plot = scatter(X[:, 4], augmented_residual; xlabel = "feature 4",
    ylabel = "residual", label = false, alpha = 0.65,
    title = "After adding the justified feature")
hline!(residual_plot, [0.0]; color = :black, linewidth = 2, label = false)
plot(cv_plot, residual_plot; layout = (1, 2), size = (1000, 400))


## 4. Check the computational contract

These checks protect the workflow's important claims: every observation appears in exactly one validation fold, the folds are reproducible, and the feature suggested by the residual pattern improves cross-validated error, not merely training error.


In [ ]:
@testset "cross-validation and residual-guided revision" begin
    @test sort(vcat(baseline_cv.folds...)) == collect(1:n)
    @test length(unique(vcat(baseline_cv.folds...))) == n
    @test baseline_cv.folds == kfold_indices(n, 6; seed = 5800)
    @test minimum(augmented_cv.mean_rmse) < 0.5 * minimum(baseline_cv.mean_rmse)
    @test augmented_metrics.rmse < baseline_metrics.rmse
    @test abs(cor(augmented_residual, quadratic_feature)) <
        abs(cor(baseline_residual, quadratic_feature))
end


## Modeling conclusion

The baseline model's validation error established that regularization alone could not repair a missing feature. Residual structure then supplied a specific model revision, and a fresh cross-validation run tested whether that revision generalized. Because the same data informed the residual diagnosis and the comparison, a final performance claim would still require an untouched test set or nested cross-validation.
